In [78]:
import pandas as pd
from collections import Counter
from graphviz import Digraph
import os

# 1. Thiết lập thư mục đầu ra
output_dir = "../results/phase_drift_analysis"
os.makedirs(output_dir, exist_ok=True)

# 2. Load và tiền xử lý dữ liệu
df = pd.read_csv("../data/bpi-challenge-2017/bpi_2017_cleaned.csv")
df = df[df["lifecycle:transition"] == "complete"]
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], format='ISO8601')
df['year'] = df['time:timestamp'].dt.year
df['week'] = df['time:timestamp'].dt.isocalendar().week
df = df.sort_values(['case:concept:name', 'time:timestamp'])

def build_dfg(df_sub):
    edges = Counter()
    for _, group in df_sub.groupby('case:concept:name'):
        activities = list(group['concept:name'])
        for i in range(len(activities) - 1):
            edges[(activities[i], activities[i+1])] += 1
    return edges

def draw_dfg(edges, title, file_path, threshold=5):
    dot = Digraph(comment=title)
    dot.attr(rankdir='LR', nodesep='0.4', ranksep='0.8', fontsize='12')
    
    # Lọc cạnh để sơ đồ nhỏ và rõ ràng
    edges = {k: v for k, v in edges.items() if v >= threshold}
    nodes = set([n for e in edges.keys() for n in e])
    
    for n in nodes:
        # Rút gọn tên để sơ đồ không bị quá rộng
        short_name = n.replace('Application', 'App').replace('Accepted', 'Acc')
        dot.node(n, short_name, shape='box', style='rounded,filled', fillcolor='#f9f9f9')
        
    for (src, tgt), freq in edges.items():
        dot.edge(src, tgt, label=str(freq), penwidth=str(0.5 + freq/200))
    
    dot.render(file_path, format='png', cleanup=True)

# 3. ĐỊNH NGHĨA 5 PHASE RỜI NHAU (DISJOINT PHASES)
# Cách chia này giúp so sánh "Trước và Sau" của từng sự kiện một cách tuyệt đối
phases = {
    "Phase_1_W1_18": df[df['week'] < 19],                            # Trước sự kiện 1
    "Phase_2_W19_23": df[(df['week'] >= 19) & (df['week'] < 24)],    # Sau sự kiện 1 (Tuần 19)
    "Phase_3_W24":    df[(df['week'] >= 24) & (df['week'] < 25)],    # Sự kiện bản lề (Tuần 24)
    "Phase_4_W25_48": df[(df['week'] >= 25) & (df['week'] < 49)],    # Sau sự kiện 2 (Tuần 25)
    "Phase_5_W49_End": df[df['week'] >= 49]                          # Cuối năm
}

# 4. Thực thi vẽ sơ đồ
for phase_name, subset in phases.items():
    print(f"Processing {phase_name}...")
    phase_path = os.path.join(output_dir, phase_name)
    os.makedirs(phase_path, exist_ok=True)
    
    for app_type in ['New credit', 'Limit raise']:
        app_label = app_type.replace(' ', '_').lower()
        app_subset = subset[subset['case:ApplicationType'] == app_type]
        
        if not app_subset.empty:
            edges = build_dfg(app_subset)
            # Threshold thấp vì dữ liệu mỗi phase đã bị chia nhỏ, giúp soi kỹ thay đổi
            draw_dfg(edges, f"{phase_name} - {app_type}", 
                     os.path.join(phase_path, f"{app_label}_dfg"),
                     threshold=15)

print(f"\n--- XONG! Sơ đồ đã được lưu tại: {output_dir} ---")

Processing Phase_1_W1_18...
Processing Phase_2_W19_23...
Processing Phase_3_W24...
Processing Phase_4_W25_48...
Processing Phase_5_W49_End...

--- XONG! Sơ đồ đã được lưu tại: ../results/phase_drift_analysis ---


In [9]:
import pandas as pd
import os
from graphviz import Digraph

# =========================================
# CONFIG
# =========================================
INPUT_FILE = "../data/bpi-challenge-2017/bpi_2017_cleaned.csv"
OUTPUT_DIR = "../results/trace_analysis"

# Nếu bạn có ID dạng số → sẽ tự convert
TARGET_CASE_NUMS = [19576, 24076]

# =========================================
# 1. LOAD DATA
# =========================================
print("📥 Loading data...")
df = pd.read_csv(INPUT_FILE)

# Convert timestamp
df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], errors="coerce")

# Drop lỗi timestamp
df = df.dropna(subset=["time:timestamp"])

# =========================================
# 2. PREPARE OUTPUT FOLDER
# =========================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================
# 3. LẤY DANH SÁCH CASE ID
# =========================================
case_ids = df["case:concept:name"].unique()

print(f"🔎 Total cases: {len(case_ids)}")

# Convert từ số → đúng ID
TARGET_CASES = []
for i in TARGET_CASE_NUMS:
    if i - 1 < len(case_ids):
        TARGET_CASES.append(case_ids[i - 1])
    else:
        print(f"⚠️ Case index {i} vượt quá dữ liệu")

print("🎯 Target cases:", TARGET_CASES)

# =========================================
# 4. FILTER DATA
# =========================================
df_subset = df[df["case:concept:name"].isin(TARGET_CASES)]
df_subset = df_subset.sort_values(["case:concept:name", "time:timestamp"])

print("📊 Số dòng sau khi lọc:", len(df_subset))

# =========================================
# 5. ANALYSIS FUNCTION
# =========================================
def analyze_trace(group, case_id):
    print(f"\n==============================")
    print(f"TRACE {case_id}")
    print(f"==============================")

    # ---- FLOW ----
    flow = [
        f"{row['concept:name']}({row['lifecycle:transition']})"
        for _, row in group.iterrows()
    ]

    print("\n🔁 Flow:")
    print(" → ".join(flow))

    # ---- LOOP ----
    activity_full = group["concept:name"] + "(" + group["lifecycle:transition"] + ")"
    counts = activity_full.value_counts()
    loops = counts[counts > 1]

    print("\n🔄 Loop:")
    if len(loops) > 0:
        print(loops)
    else:
        print("No loop")

    # ---- TIME CHECK ----
    if not group["time:timestamp"].is_monotonic_increasing:
        print("⚠️ Timestamp ERROR")
    else:
        print("⏱ Timestamp OK")

    # ---- BUSINESS LOGIC ----
    if "A_Accepted" not in group["concept:name"].values:
        print("⚠️ Không có A_Accepted (case có thể fail)")

    if "O_Sent" not in group["concept:name"].values:
        print("⚠️ Không có O_Sent (chưa gửi offer)")


# =========================================
# 6. GRAPH FUNCTION
# =========================================
def draw_trace_graph(group, case_id):
    dot = Digraph(comment=f"Trace {case_id}", format="png")

    events = group.reset_index(drop=True)

    # ---- CREATE NODES ----
    for i, row in events.iterrows():
        activity = row["concept:name"]
        lifecycle = row["lifecycle:transition"]
        timestamp = row["time:timestamp"]

        label = f"{activity}\n({lifecycle})\n{timestamp}"

        # Default
        color = "black"

        # ---- Highlight ----
        if lifecycle == "withdraw":
            color = "red"        # cancel
        elif lifecycle == "schedule":
            color = "blue"       # scheduled
        elif lifecycle == "complete":
            color = "green"      # done

        dot.node(str(i), label, color=color)

    # ---- CREATE EDGES ----
    for i in range(len(events) - 1):
        dot.edge(str(i), str(i + 1))

    # ---- SAVE ----
    safe_case_id = case_id.replace("/", "_")
    output_path = os.path.join(OUTPUT_DIR, safe_case_id)

    dot.render(output_path, view=False)
    print(f"🖼 Saved: {output_path}.png")


# =========================================
# 7. RUN
# =========================================
print("\n🚀 Start processing...")

for case_id, group in df_subset.groupby("case:concept:name"):
    analyze_trace(group, case_id)
    draw_trace_graph(group, case_id)

print("\n✅ DONE!")

📥 Loading data...
🔎 Total cases: 31509
🎯 Target cases: ['Application_1943497450', 'Application_1138441532']
📊 Số dòng sau khi lọc: 85

🚀 Start processing...

TRACE Application_1138441532

🔁 Flow:
A_Create Application(complete) → A_Submitted(complete) → W_Handle leads(schedule) → W_Handle leads(withdraw) → W_Complete application(schedule) → A_Concept(complete) → W_Complete application(start) → W_Complete application(suspend) → W_Complete application(resume) → W_Complete application(suspend) → W_Complete application(resume) → W_Complete application(suspend) → W_Complete application(resume) → A_Accepted(complete) → O_Create Offer(complete) → O_Created(complete) → O_Sent (mail and online)(complete) → W_Complete application(complete) → W_Call after offers(schedule) → W_Call after offers(start) → A_Complete(complete) → W_Call after offers(suspend) → W_Call after offers(resume) → W_Call after offers(suspend) → W_Call after offers(ate_abort) → W_Validate application(schedule) → W_Validate appl